# Pipeline Project

You will be using the provided data to create a machine learning model pipeline.

You must handle the data appropriately in your pipeline to predict whether an
item is recommended by a customer based on their review.
Note the data includes numerical, categorical, and text data.

You should ensure you properly train and evaluate your model.

## The Data

The dataset has been anonymized and cleaned of missing values.

There are 8 features for to use to predict whether a customer recommends or does
not recommend a product.
The `Recommended IND` column gives whether a customer recommends the product
where `1` is recommended and a `0` is not recommended.
This is your model's target/

The features can be summarized as the following:

- **Clothing ID**: Integer Categorical variable that refers to the specific piece being reviewed.
- **Age**: Positive Integer variable of the reviewers age.
- **Title**: String variable for the title of the review.
- **Review Text**: String variable for the review body.
- **Positive Feedback Count**: Positive Integer documenting the number of other customers who found this review positive.
- **Division Name**: Categorical name of the product high level division.
- **Department Name**: Categorical name of the product department name.
- **Class Name**: Categorical name of the product class name.

The target:
- **Recommended IND**: Binary variable stating where the customer recommends the product where 1 is recommended, 0 is not recommended.

## Load Data

In [1]:
import pandas as pd

# Load data
df = pd.read_csv(
    'data/reviews.csv',
)

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18442 entries, 0 to 18441
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Clothing ID              18442 non-null  int64 
 1   Age                      18442 non-null  int64 
 2   Title                    18442 non-null  object
 3   Review Text              18442 non-null  object
 4   Positive Feedback Count  18442 non-null  int64 
 5   Division Name            18442 non-null  object
 6   Department Name          18442 non-null  object
 7   Class Name               18442 non-null  object
 8   Recommended IND          18442 non-null  int64 
dtypes: int64(4), object(5)
memory usage: 1.3+ MB


,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name,Recommended IND
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses,0
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants,1
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses,1
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses,0
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits,1


## Preparing features (`X`) & target (`y`)

In [2]:
data = df

# separate features from labels
X = data.drop('Recommended IND', axis=1)
y = data['Recommended IND'].copy()

print('Labels:', y.unique())
print('Features:')
display(X.head())

Labels: [0 1]
Features:


,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits


In [3]:
# Split data into train and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    shuffle=True,
    random_state=27,
)

# Your Work

## Data Exploration

In [5]:
# EDA
print (df.shape)
print(df['Recommended IND'].value_counts(normalize=True))
print(df.describe(include='all').T)

(18442, 9)
Recommended IND
1    0.816235
0    0.183765
Name: proportion, dtype: float64
                           count unique  \
Clothing ID              18442.0    NaN   
Age                      18442.0    NaN   
Title                      18442  13142   
Review Text                18442  18439   
Positive Feedback Count  18442.0    NaN   
Division Name              18442      2   
Department Name            18442      6   
Class Name                 18442     14   
Recommended IND          18442.0    NaN   

                                                                       top  \
Clothing ID                                                            NaN   
Age                                                                    NaN   
Title                                                             Love it!   
Review Text              I bought this shirt at the store and after goi...   
Positive Feedback Count                                                NaN   
Division Name

In [6]:
df.columns

Index(['Clothing ID', 'Age', 'Title', 'Review Text', 'Positive Feedback Count',
       'Division Name', 'Department Name', 'Class Name', 'Recommended IND'],
      dtype='object')

## Building Pipeline

In [8]:
# Feature groups
num_features = ['Age', 'Positive Feedback Count']
cat_features = ['Clothing ID', 'Division Name', 'Department Name', 'Class Name']
text_features = ['Title', 'Review Text']

### Numerical Features Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

num_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='median'),
    ),
    (
        'scaler',
        MinMaxScaler(),
    ),
])

### Categorical Features Pipeline

In [11]:
from sklearn.preprocessing import OneHotEncoder

cat_pipeline = Pipeline([   
    (
        'imputer',
        SimpleImputer(
            strategy='most_frequent',
        )
    ),
    (
        'cat_encoder',
        OneHotEncoder(
            sparse_output=False,
            handle_unknown='ignore',
        )
    ),
])

### Text Feature Pipeline

#### Custom `Transformer`: Count Characters

In [13]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

# Combine Title and Review Text
class TextCombiner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        # Return as a 1D NumPy array for downstream NLP transformers
        return (X.iloc[:, 0].fillna('') + ' ' + X.iloc[:, 1].fillna('')).values 
    
# Create CountCharacter()
class CountCharacter(BaseEstimator, TransformerMixin):
    def __init__(self, character: str):
        self.character = character

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.array([[text.count(self.character)] for text in X])

In [18]:
from sklearn.pipeline import FeatureUnion

# create a pipeline
character_counts_pipeline = Pipeline([
    ('combine_text', TextCombiner()),
    ('features', FeatureUnion([
        ('count_spaces', CountCharacter(character=' ')),
        ('count_exclamations', CountCharacter(character='!')),
        ('count_question_marks', CountCharacter(character='?')),
    ])),
])
character_counts_pipeline

Pipeline(steps=[('combine_text', TextCombiner()),
                ('features',
                 FeatureUnion(transformer_list=[('count_spaces',
                                                 CountCharacter(character=' ')),
                                                ('count_exclamations',
                                                 CountCharacter(character='!')),
                                                ('count_question_marks',
                                                 CountCharacter(character='?'))]))])

#### Custom `Transformer`: TF-IDF

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

# create a pipeline
text_tfidf_pipeline = Pipeline([
    ('combine_text', TextCombiner()),
    ('tfidf', TfidfVectorizer(
        stop_words='english'
    )),
])
text_tfidf_pipeline

Pipeline(steps=[('combine_text', TextCombiner()),
                ('tfidf', TfidfVectorizer(stop_words='english'))])

### Combine Feature Engineering Pipelines

In [20]:
from sklearn.compose import ColumnTransformer

feature_engineering = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features),
    ('character_counts', character_counts_pipeline, text_features),
    ('tfidf_text', text_tfidf_pipeline, text_features),
])
feature_engineering

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Age', 'Positive Feedback Count']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('cat_encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Clothing ID', 'Division Name',
                                  'Departm...
                                                  FeatureUnion(transformer_list=[('count_spaces',
                                                                                  CountCharacter(character=' ')),
                                                                                 ('count_exclamations',
                                                                                  CountCharacter(character='!')),
                                                                                 ('count_question_marks',
                                                                                  CountCharacter(character='?'))]))]),
                                 ['Title', 'Review Text']),
                                ('tfidf_text',
                                 Pipeline(steps=[('combine_text',
                                                  TextCombiner()),
                                                 ('tfidf',
                                                  TfidfVectorizer(stop_words='english'))]),
                                 ['Title', 'Review Text'])])

## Training Pipeline

## Fine-Tuning Pipeline